## 1. Environment Setup and Data File Validation


In [ ]:
from pathlib import Path
import polars as pl

DATA_PATH = Path.cwd() / "2019-Oct.csv"

print("Current directory:", Path.cwd())
print("File exists:", DATA_PATH.exists())

if DATA_PATH.exists():
    print(f"File size: {DATA_PATH.stat().st_size / (1024**3):.2f} GB")
else:
    print("")

In [1]:
import sys
import polars as pl

print("Python:", sys.executable)
print("Polars version:", pl.__version__)

Python: d:\Samsung\Graduation Project\Data\Old Data\.venv\Scripts\python.exe
Polars version: 2.0.0


#### 2. Dataset File Validation

In this step, we verify that the original CSV file exists in the notebook's working directory and inspect its file size.

- `Path.cwd()` displays the current working directory.
- `DATA_PATH.exists()` checks whether the dataset file exists.
- `DATA_PATH.stat().st_size` retrieves the file size in bytes.

**Objective:** Confirm that the notebook can access the original dataset before starting the exploration.

**Note:** The original dataset will remain unchanged.

In [2]:
from pathlib import Path

DATA_PATH = Path.cwd() / "2019-Oct.csv"

print("Working directory:", Path.cwd())
print("Dataset exists:", DATA_PATH.exists())

if DATA_PATH.exists():
    size_gb = DATA_PATH.stat().st_size / (1024 ** 3)
    print(f"Dataset size: {size_gb:.2f} GB")
else:
    print("Dataset not found. Check the file name and notebook directory.")

Working directory: d:\Samsung\Graduation Project\Data\Old Data
Dataset exists: True
Dataset size: 5.28 GB


## 3. Schema Inspection

In this step, we inspect the dataset's column names and inferred data types using Polars Lazy API.

- `pl.scan_csv()` creates a lazy query without immediately loading the entire dataset.
- `infer_schema_length=10000` uses the first 10,000 rows to infer column types.
- `collect_schema()` retrieves the schema without collecting all the dataset rows.

**Objective:** Understand the dataset structure before checking data quality, event types, missing values, and the total number of records.

**Note:** Schema inference may need adjustment if some columns contain inconsistent values.

In [3]:
import polars as pl

df = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"]
)

schema = df.collect_schema()

print("Column names:")
print(schema.names())

print("\nData types:")
print(schema)

Column names:
['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']

Data types:
Schema({'event_time': String, 'event_type': String, 'product_id': Int64, 'category_id': Int64, 'category_code': String, 'brand': String, 'price': Float64, 'user_id': Int64, 'user_session': String})


## 4. Dataset Size and Event Type Distribution

In this step, we calculate the total number of records and count the occurrences of each event type.

- `pl.len()` counts the dataset rows.
- `group_by("event_type").len()` counts records for each event type.
- `sort()` arranges event types by frequency.

**Objective:** Establish the dataset's actual size and identify which event types exist before defining the anomaly scenarios.

**Note:** These operations scan the large CSV file, so execution may take some time. The original dataset is not modified.

In [4]:
# 1. Count all records
total_rows = df.select(
    pl.len().alias("total_rows")
).collect(engine="streaming").item()

print(f"Total records: {total_rows:,}")

# 2. Count records for each event type
event_counts = (
    df.group_by("event_type")
    .len(name="records")
    .sort("records", descending=True)
    .collect(engine="streaming")
)

print("\nEvent type distribution:")
print(event_counts)

# 3. Calculate the percentage of each event type
event_percentages = event_counts.with_columns(
    (
        pl.col("records") / total_rows * 100
    ).round(2).alias("percentage")
)

print("\nEvent type percentages:")
print(event_percentages)

Total records: 42,448,764

Event type distribution:
shape: (3, 2)
┌────────────┬──────────┐
│ event_type ┆ records  │
│ ---        ┆ ---      │
│ str        ┆ u32      │
╞════════════╪══════════╡
│ view       ┆ 40779399 │
│ cart       ┆ 926516   │
│ purchase   ┆ 742849   │
└────────────┴──────────┘

Event type percentages:
shape: (3, 3)
┌────────────┬──────────┬────────────┐
│ event_type ┆ records  ┆ percentage │
│ ---        ┆ ---      ┆ ---        │
│ str        ┆ u32      ┆ f64        │
╞════════════╪══════════╪════════════╡
│ view       ┆ 40779399 ┆ 96.07      │
│ cart       ┆ 926516   ┆ 2.18       │
│ purchase   ┆ 742849   ┆ 1.75       │
└────────────┴──────────┴────────────┘


## 5. Missing Values Analysis

In this step, we count missing values in each column without modifying the dataset.

**Objective:** Identify data completeness issues before designing the synthetic anomaly scenarios.

**Important:** This is a read-only analysis. No columns, data types, or original values will be changed.

In [5]:
missing_counts = (
    df.select([
        pl.col(column).null_count().alias(column)
        for column in df.collect_schema().names()
    ])
    .collect(engine="streaming")
)

print("Missing values per column:")
print(missing_counts)

Missing values per column:
shape: (1, 9)
┌────────────┬────────────┬────────────┬─────────────┬───┬─────────┬───────┬─────────┬─────────────┐
│ event_time ┆ event_type ┆ product_id ┆ category_id ┆ … ┆ brand   ┆ price ┆ user_id ┆ user_sessio │
│ ---        ┆ ---        ┆ ---        ┆ ---         ┆   ┆ ---     ┆ ---   ┆ ---     ┆ n           │
│ u32        ┆ u32        ┆ u32        ┆ u32         ┆   ┆ u32     ┆ u32   ┆ u32     ┆ ---         │
│            ┆            ┆            ┆             ┆   ┆         ┆       ┆         ┆ u32         │
╞════════════╪════════════╪════════════╪═════════════╪═══╪═════════╪═══════╪═════════╪═════════════╡
│ 0          ┆ 0          ┆ 0          ┆ 0           ┆ … ┆ 6113008 ┆ 0     ┆ 0       ┆ 2           │
└────────────┴────────────┴────────────┴─────────────┴───┴─────────┴───────┴─────────┴─────────────┘


## 6. Detailed Missing Values Report

This step formats the missing-value counts into a readable table with one row per column.

**Objective:** Review missing values across all nine original columns without changing the schema or modifying the source data.

In [6]:
missing_report = (
    df.select([
        pl.col(column).null_count().alias(column)
        for column in df.collect_schema().names()
    ])
    .collect(engine="streaming")
    .transpose(
        include_header=True,
        header_name="column",
        column_names=["missing_values"]
    )
)

print(missing_report)

shape: (9, 2)
┌───────────────┬────────────────┐
│ column        ┆ missing_values │
│ ---           ┆ ---            │
│ str           ┆ u32            │
╞═══════════════╪════════════════╡
│ event_time    ┆ 0              │
│ event_type    ┆ 0              │
│ product_id    ┆ 0              │
│ category_id   ┆ 0              │
│ category_code ┆ 13515609       │
│ brand         ┆ 6113008        │
│ price         ┆ 0              │
│ user_id       ┆ 0              │
│ user_session  ┆ 2              │
└───────────────┴────────────────┘


## 7. Event Time Range Analysis

In this step, we inspect the minimum and maximum event timestamps in the original dataset.

**Objective:** Understand the time coverage before selecting a synthetic promotional period.

**Important:** The `event_time` column remains a string. This step does not modify the schema or source data.

In [7]:
time_range = df.select([
    pl.col("event_time").min().alias("first_event"),
    pl.col("event_time").max().alias("last_event")
]).collect(engine="streaming")

print(time_range)

shape: (1, 2)
┌─────────────────────────┬─────────────────────────┐
│ first_event             ┆ last_event              │
│ ---                     ┆ ---                     │
│ str                     ┆ str                     │
╞═════════════════════════╪═════════════════════════╡
│ 2019-10-01 00:00:00 UTC ┆ 2019-10-31 23:59:59 UTC │
└─────────────────────────┴─────────────────────────┘


## 8. Original Duplicate Records Analysis

In this step, we check whether the original dataset contains fully identical records across all nine columns.

**Objective:** Establish a baseline for the Duplicate Events anomaly scenario.

**Method:** Group records by all original columns and identify groups occurring more than once.

**Important:** This is a read-only operation. No schema or source records will be changed.

In [8]:
columns = df.collect_schema().names()

duplicate_groups = (
    df.group_by(columns)
    .len(name="occurrences")
    .filter(pl.col("occurrences") > 1)
)

duplicate_summary = duplicate_groups.select([
    pl.len().alias("duplicate_groups"),
    (pl.col("occurrences") - 1).sum().alias("extra_duplicate_rows")
]).collect(engine="streaming")

print(duplicate_summary)

shape: (1, 2)
┌──────────────────┬──────────────────────┐
│ duplicate_groups ┆ extra_duplicate_rows │
│ ---              ┆ ---                  │
│ u32              ┆ u32                  │
╞══════════════════╪══════════════════════╡
│ 18099            ┆ 30220                │
└──────────────────┴──────────────────────┘


## 9. Daily Event Distribution

In this step, we analyze the daily distribution of each event type throughout October 2019.

- Extract the date portion from `event_time` as a temporary expression.
- Group records by date and event type.
- Count the number of records for each combination.
- Pivot the results into daily counts for `view`, `cart`, and `purchase`.

**Objective:** Understand daily activity patterns and identify potential periods for simulating a promotional purchase surge.

**Important:** The original `event_time` column remains a string. No columns or source records are modified.

In [9]:
daily_events = (
    df.with_columns(
        pl.col("event_time")
        .str.slice(0, 10)
        .alias("event_date")
    )
    .group_by(["event_date", "event_type"])
    .len(name="records")
    .collect(engine="streaming")
)

daily_summary = (
    daily_events
    .pivot(
        on="event_type",
        index="event_date",
        values="records",
        aggregate_function="sum"
    )
    .sort("event_date")
    .fill_null(0)
)

print(daily_summary)

shape: (31, 4)
┌────────────┬───────┬─────────┬──────────┐
│ event_date ┆ cart  ┆ view    ┆ purchase │
│ ---        ┆ ---   ┆ ---     ┆ ---      │
│ str        ┆ u32   ┆ u32     ┆ u32      │
╞════════════╪═══════╪═════════╪══════════╡
│ 2019-10-01 ┆ 16658 ┆ 1208280 ┆ 19307    │
│ 2019-10-02 ┆ 17268 ┆ 1154591 ┆ 19469    │
│ 2019-10-03 ┆ 19323 ┆ 1088725 ┆ 19255    │
│ 2019-10-04 ┆ 43829 ┆ 1346320 ┆ 27041    │
│ 2019-10-05 ┆ 35497 ┆ 1271348 ┆ 23494    │
│ …          ┆ …     ┆ …       ┆ …        │
│ 2019-10-27 ┆ 34256 ┆ 1330894 ┆ 23403    │
│ 2019-10-28 ┆ 17838 ┆ 1221930 ┆ 21112    │
│ 2019-10-29 ┆ 17455 ┆ 1190081 ┆ 20374    │
│ 2019-10-30 ┆ 16996 ┆ 1172332 ┆ 20817    │
│ 2019-10-31 ┆ 16798 ┆ 1208582 ┆ 20099    │
└────────────┴───────┴─────────┴──────────┘


## 10. Highest Purchase Activity Days

In this step, we rank all days by their purchase event counts.

**Objective:** Identify the days with the highest purchase activity and compare them with the daily baseline before designing a synthetic promotional surge.

**Note:** This analysis is descriptive only and does not establish the cause of any observed increase.

In [10]:
purchase_ranking = daily_summary.sort(
    "purchase",
    descending=True
)

print(purchase_ranking)

shape: (31, 4)
┌────────────┬───────┬─────────┬──────────┐
│ event_date ┆ cart  ┆ view    ┆ purchase │
│ ---        ┆ ---   ┆ ---     ┆ ---      │
│ str        ┆ u32   ┆ u32     ┆ u32      │
╞════════════╪═══════╪═════════╪══════════╡
│ 2019-10-16 ┆ 45279 ┆ 1445717 ┆ 31394    │
│ 2019-10-13 ┆ 24833 ┆ 1584677 ┆ 29561    │
│ 2019-10-14 ┆ 44858 ┆ 1381439 ┆ 28405    │
│ 2019-10-17 ┆ 40378 ┆ 1338311 ┆ 28318    │
│ 2019-10-04 ┆ 43829 ┆ 1346320 ┆ 27041    │
│ …          ┆ …     ┆ …       ┆ …        │
│ 2019-10-29 ┆ 17455 ┆ 1190081 ┆ 20374    │
│ 2019-10-31 ┆ 16798 ┆ 1208582 ┆ 20099    │
│ 2019-10-02 ┆ 17268 ┆ 1154591 ┆ 19469    │
│ 2019-10-01 ┆ 16658 ┆ 1208280 ┆ 19307    │
│ 2019-10-03 ┆ 19323 ┆ 1088725 ┆ 19255    │
└────────────┴───────┴─────────┴──────────┘


## 11. User Session Activity Analysis

In this step, we calculate the number of events recorded in each user session and summarize the session-length distribution.

- Group events by `user_session`.
- Count the events within each session.
- Calculate summary statistics to understand typical and unusually active sessions.

**Objective:** Establish a baseline for detecting unusually repetitive or highly active sessions.

**Important:** The original dataset and schema remain unchanged. Sessions with missing identifiers are excluded from this session-level analysis only.

In [11]:
session_lengths = (
    df.filter(pl.col("user_session").is_not_null())
    .group_by("user_session")
    .len(name="event_count")
)

session_summary = session_lengths.select([
    pl.len().alias("total_sessions"),
    pl.col("event_count").min().alias("min_events"),
    pl.col("event_count").median().alias("median_events"),
    pl.col("event_count").quantile(0.90).alias("p90_events"),
    pl.col("event_count").quantile(0.95).alias("p95_events"),
    pl.col("event_count").quantile(0.99).alias("p99_events"),
    pl.col("event_count").max().alias("max_events"),
]).collect(engine="streaming")

print(session_summary)

shape: (1, 7)
┌────────────────┬────────────┬───────────────┬────────────┬────────────┬────────────┬────────────┐
│ total_sessions ┆ min_events ┆ median_events ┆ p90_events ┆ p95_events ┆ p99_events ┆ max_events │
│ ---            ┆ ---        ┆ ---           ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│ u32            ┆ u32        ┆ f64           ┆ f64        ┆ f64        ┆ f64        ┆ u32        │
╞════════════════╪════════════╪═══════════════╪════════════╪════════════╪════════════╪════════════╡
│ 9244421        ┆ 1          ┆ 2.0           ┆ 10.0       ┆ 16.0       ┆ 32.0       ┆ 1159       │
└────────────────┴────────────┴───────────────┴────────────┴────────────┴────────────┴────────────┘


## 12. Most Active User Sessions

This step identifies the 20 sessions with the highest event counts and breaks down their activity by event type.

**Objective:** Investigate unusually active sessions before defining a synthetic bot-like crawling pattern.

**Method:** Aggregate events by session and event type, then summarize the results for the most active sessions.

**Important:** This is read-only analysis. The original schema and source data remain unchanged.

In [14]:
session_event_counts = (
    df.filter(pl.col("user_session").is_not_null())
    .group_by(["user_session", "event_type"])
    .len(name="event_count")
)

top_sessions = (
    session_event_counts
    .group_by("user_session")
    .agg([
        pl.col("event_count")
        .filter(pl.col("event_type") == "view")
        .sum()
        .alias("view"),

        pl.col("event_count")
        .filter(pl.col("event_type") == "cart")
        .sum()
        .alias("cart"),

        pl.col("event_count")
        .filter(pl.col("event_type") == "purchase")
        .sum()
        .alias("purchase"),

        pl.col("event_count").sum().alias("total_events"),
    ])
    .sort("total_events", descending=True)
    .head(20)
    .collect(engine="streaming")
)

print(top_sessions)

shape: (20, 5)
┌─────────────────────────────────┬──────┬──────┬──────────┬──────────────┐
│ user_session                    ┆ view ┆ cart ┆ purchase ┆ total_events │
│ ---                             ┆ ---  ┆ ---  ┆ ---      ┆ ---          │
│ str                             ┆ u32  ┆ u32  ┆ u32      ┆ u32          │
╞═════════════════════════════════╪══════╪══════╪══════════╪══════════════╡
│ fb075266-182d-4c11-b5f7-4e4dcd… ┆ 1158 ┆ 1    ┆ 0        ┆ 1159         │
│ cfb90a35-9575-495c-b6aa-48ddca… ┆ 1137 ┆ 0    ┆ 0        ┆ 1137         │
│ 2183f046-46f1-4ff6-96ef-f74986… ┆ 584  ┆ 0    ┆ 0        ┆ 584          │
│ b2101293-44c1-4814-836a-94b0c0… ┆ 564  ┆ 0    ┆ 0        ┆ 564          │
│ e9d2b8ad-3e69-47f2-8b8a-1fadfc… ┆ 425  ┆ 0    ┆ 0        ┆ 425          │
│ …                               ┆ …    ┆ …    ┆ …        ┆ …            │
│ 359dc98c-6d30-4645-8f68-2999a5… ┆ 336  ┆ 0    ┆ 0        ┆ 336          │
│ 0deca816-5a12-45d8-8474-d28c2b… ┆ 325  ┆ 1    ┆ 0        ┆ 326         

## 13. Repeated Product Views Within Sessions

This step investigates whether highly active sessions repeatedly view the same products or browse many different products.

**Objective:** Distinguish repeated-view patterns from broad browsing activity before defining bot-like synthetic anomalies.

**Method:** For each session, count view events and distinct viewed products, then rank sessions by repeated-view activity.

**Important:** The analysis is read-only and does not modify the original schema or source file.

In [15]:
view_patterns = (
    df.filter(
        pl.col("user_session").is_not_null()
        & (pl.col("event_type") == "view")
    )
    .group_by("user_session")
    .agg([
        pl.len().alias("view_events"),
        pl.col("product_id").n_unique().alias("distinct_products"),
    ])
    .with_columns(
        (
            pl.col("view_events") - pl.col("distinct_products")
        ).alias("repeated_view_events")
    )
    .sort("repeated_view_events", descending=True)
    .head(20)
    .collect(engine="streaming")
)

print(view_patterns)

shape: (20, 4)
┌─────────────────────────────────┬─────────────┬───────────────────┬──────────────────────┐
│ user_session                    ┆ view_events ┆ distinct_products ┆ repeated_view_events │
│ ---                             ┆ ---         ┆ ---               ┆ ---                  │
│ str                             ┆ u32         ┆ u32               ┆ u32                  │
╞═════════════════════════════════╪═════════════╪═══════════════════╪══════════════════════╡
│ fb075266-182d-4c11-b5f7-4e4dcd… ┆ 1158        ┆ 290               ┆ 868                  │
│ b2101293-44c1-4814-836a-94b0c0… ┆ 564         ┆ 88                ┆ 476                  │
│ 2183f046-46f1-4ff6-96ef-f74986… ┆ 584         ┆ 218               ┆ 366                  │
│ cfb90a35-9575-495c-b6aa-48ddca… ┆ 1137        ┆ 868               ┆ 269                  │
│ 0deca816-5a12-45d8-8474-d28c2b… ┆ 325         ┆ 64                ┆ 261                  │
│ …                               ┆ …           ┆ …    

## 14. Event Sequence Inspection

This step samples sessions containing multiple event types and inspects their events in chronological order.

**Objective:** Understand observed event sequences before defining synthetic impossible-sequence anomalies.

**Method:** Select a small sample of sessions with multiple event types, then sort their events by the original `event_time` string.

**Important:** The original `event_time` column remains a string. No source records or schema definitions are modified.

In [16]:
sample_sessions = (
    df.filter(pl.col("user_session").is_not_null())
    .group_by("user_session")
    .agg(pl.col("event_type").n_unique().alias("event_type_count"))
    .filter(pl.col("event_type_count") >= 2)
    .head(5)
    .select("user_session")
    .collect(engine="streaming")
)

sample_session_ids = sample_sessions["user_session"].to_list()

sequence_sample = (
    df.filter(pl.col("user_session").is_in(sample_session_ids))
    .select([
        "user_session",
        "event_time",
        "event_type",
        "product_id",
    ])
    .sort(["user_session", "event_time"])
    .collect(engine="streaming")
)

print(sequence_sample)

shape: (75, 4)
┌─────────────────────────────────┬─────────────────────────┬────────────┬────────────┐
│ user_session                    ┆ event_time              ┆ event_type ┆ product_id │
│ ---                             ┆ ---                     ┆ ---        ┆ ---        │
│ str                             ┆ str                     ┆ str        ┆ i64        │
╞═════════════════════════════════╪═════════════════════════╪════════════╪════════════╡
│ 1d99c050-4c43-41b2-9c05-ce63eb… ┆ 2019-10-01 06:43:04 UTC ┆ view       ┆ 1005098    │
│ 1d99c050-4c43-41b2-9c05-ce63eb… ┆ 2019-10-01 06:44:53 UTC ┆ purchase   ┆ 1005098    │
│ 82b0041f-db07-479f-bb37-acf492… ┆ 2019-10-01 07:07:06 UTC ┆ view       ┆ 1802019    │
│ 82b0041f-db07-479f-bb37-acf492… ┆ 2019-10-01 07:07:48 UTC ┆ purchase   ┆ 1802019    │
│ 82b0041f-db07-479f-bb37-acf492… ┆ 2019-10-01 07:08:19 UTC ┆ view       ┆ 1802019    │
│ …                               ┆ …                       ┆ …          ┆ …          │
│ bb7a4ae7-870c-4

In [17]:
# Find sessions where a product is purchased without a preceding view
sample_ids = sample_session_ids

sequence_check = (
    df.filter(pl.col("user_session").is_in(sample_ids))
    .select([
        "user_session",
        "event_time",
        "event_type",
        "product_id",
    ])
    .sort(["user_session", "event_time"])
    .collect(engine="streaming")
)

# Display the events in chronological order for manual inspection
print(sequence_check)

shape: (75, 4)
┌─────────────────────────────────┬─────────────────────────┬────────────┬────────────┐
│ user_session                    ┆ event_time              ┆ event_type ┆ product_id │
│ ---                             ┆ ---                     ┆ ---        ┆ ---        │
│ str                             ┆ str                     ┆ str        ┆ i64        │
╞═════════════════════════════════╪═════════════════════════╪════════════╪════════════╡
│ 1d99c050-4c43-41b2-9c05-ce63eb… ┆ 2019-10-01 06:43:04 UTC ┆ view       ┆ 1005098    │
│ 1d99c050-4c43-41b2-9c05-ce63eb… ┆ 2019-10-01 06:44:53 UTC ┆ purchase   ┆ 1005098    │
│ 82b0041f-db07-479f-bb37-acf492… ┆ 2019-10-01 07:07:06 UTC ┆ view       ┆ 1802019    │
│ 82b0041f-db07-479f-bb37-acf492… ┆ 2019-10-01 07:07:48 UTC ┆ purchase   ┆ 1802019    │
│ 82b0041f-db07-479f-bb37-acf492… ┆ 2019-10-01 07:08:19 UTC ┆ view       ┆ 1802019    │
│ …                               ┆ …                       ┆ …          ┆ …          │
│ bb7a4ae7-870c-4

## 15. Impossible Sequence Inspection — Purchase Without Prior View

### Objective
Identify purchase events that have no preceding view event for the same product within the same user session.

### Method
- Extract `view` and `purchase` events from the sampled sessions.
- Match events using `user_session` and `product_id`.
- Compare `event_time` values to determine whether a view occurred before the purchase.
- Flag purchases with no preceding view for further investigation.

### Expected Outcome
A list of purchase events that have no earlier matching view in the sampled data.

### Important Notes
- These events are **potential anomalies**, not confirmed errors, because the user may have viewed the product before the recorded session or some events may be missing.
- The analysis is limited to the sampled sessions, not the full dataset.
- The original dataset and schema remain unchanged. `event_time` stays a string, and no columns are added to the source data.

In [18]:
import polars as pl

events = sequence_check

# Collect the product views for each session
views = (
    events
    .filter(pl.col("event_type") == "view")
    .select([
        "user_session",
        "product_id",
        pl.col("event_time").alias("view_time"),
    ])
)

# Get all purchases
purchases = (
    events
    .filter(pl.col("event_type") == "purchase")
    .select([
        "user_session",
        "product_id",
        pl.col("event_time").alias("purchase_time"),
    ])
)

# Match each purchase with views of the same product in the same session
purchase_check = (
    purchases
    .join(
        views,
        on=["user_session", "product_id"],
        how="left",
    )
    .with_columns(
        (
            pl.col("view_time").is_not_null()
            & (pl.col("view_time") < pl.col("purchase_time"))
        ).alias("has_prior_view")
    )
    .group_by(["user_session", "product_id", "purchase_time"])
    .agg(
        pl.col("has_prior_view").any().alias("has_prior_view")
    )
    .filter(~pl.col("has_prior_view"))
)

print(purchase_check)

shape: (0, 4)
┌──────────────┬────────────┬───────────────┬────────────────┐
│ user_session ┆ product_id ┆ purchase_time ┆ has_prior_view │
│ ---          ┆ ---        ┆ ---           ┆ ---            │
│ str          ┆ i64        ┆ str           ┆ bool           │
╞══════════════╪════════════╪═══════════════╪════════════════╡
└──────────────┴────────────┴───────────────┴────────────────┘


## 16. Daily Event Baseline Analysis

### Objective
Establish a daily baseline for ecommerce activity before injecting synthetic anomalies.

### Method
- Group events by calendar date.
- Calculate the number of `view`, `cart`, and `purchase` events per day.
- Calculate the daily cart-to-view and purchase-to-view event ratios.
- Identify daily variations that can help define realistic synthetic promotional surges.

### Expected Outcome
A daily summary table showing event volumes and event-level ratios across October 2019.

### Important Notes
- The original `event_time` column remains a string.
- Date extraction is performed inside the analysis query only.
- No source records or schema definitions are modified.
- Event-level ratios are not equivalent to unique-user conversion rates.

In [19]:
import polars as pl

daily_baseline = (
    df
    .with_columns(
        pl.col("event_time")
        .str.slice(0, 10)
        .alias("event_date")
    )
    .group_by("event_date")
    .agg([
        (pl.col("event_type") == "view")
        .sum()
        .alias("view_events"),

        (pl.col("event_type") == "cart")
        .sum()
        .alias("cart_events"),

        (pl.col("event_type") == "purchase")
        .sum()
        .alias("purchase_events"),
    ])
    .with_columns([
        (
            pl.col("cart_events") / pl.col("view_events") * 100
        ).round(3).alias("cart_to_view_pct"),

        (
            pl.col("purchase_events") / pl.col("view_events") * 100
        ).round(3).alias("purchase_to_view_pct"),

        (
            pl.col("view_events")
            + pl.col("cart_events")
            + pl.col("purchase_events")
        ).alias("total_events"),
    ])
    .sort("event_date")
)

daily_result = daily_baseline.collect(engine="streaming")

print(daily_result)

shape: (31, 7)
┌────────────┬─────────────┬─────────────┬──────────────┬──────────────┬─────────────┬─────────────┐
│ event_date ┆ view_events ┆ cart_events ┆ purchase_eve ┆ cart_to_view ┆ purchase_to ┆ total_event │
│ ---        ┆ ---         ┆ ---         ┆ nts          ┆ _pct         ┆ _view_pct   ┆ s           │
│ str        ┆ u32         ┆ u32         ┆ ---          ┆ ---          ┆ ---         ┆ ---         │
│            ┆             ┆             ┆ u32          ┆ f64          ┆ f64         ┆ u32         │
╞════════════╪═════════════╪═════════════╪══════════════╪══════════════╪═════════════╪═════════════╡
│ 2019-10-01 ┆ 1208280     ┆ 16658       ┆ 19307        ┆ 1.379        ┆ 1.598       ┆ 1244245     │
│ 2019-10-02 ┆ 1154591     ┆ 17268       ┆ 19469        ┆ 1.496        ┆ 1.686       ┆ 1191328     │
│ 2019-10-03 ┆ 1088725     ┆ 19323       ┆ 19255        ┆ 1.775        ┆ 1.769       ┆ 1127303     │
│ 2019-10-04 ┆ 1346320     ┆ 43829       ┆ 27041        ┆ 3.255        ┆ 2.0

## 17. Daily Purchase Baseline Ranking

### Objective
Rank daily purchase activity to identify high-activity and low-activity dates.

### Method
- Sort the daily summary by purchase-event count.
- Compare the highest and lowest purchase days.
- Review purchase-to-view ratios alongside purchase counts.

### Expected Outcome
A ranked summary that helps establish a reference baseline for designing a synthetic promotional purchase surge.

### Important Notes
A high purchase count does not automatically indicate an anomaly or a real promotion. The ranking is exploratory and uses observed source data only.

In [20]:
# Highest purchase-event days
top_purchase_days = daily_result.sort(
    "purchase_events",
    descending=True
).head(10)

print("Top 10 days by purchase events:")
print(
    top_purchase_days.select([
        "event_date",
        "view_events",
        "purchase_events",
        "purchase_to_view_pct",
        "total_events",
    ])
)

# Lowest purchase-event days
lowest_purchase_days = daily_result.sort(
    "purchase_events"
).head(5)

print("\nBottom 5 days by purchase events:")
print(
    lowest_purchase_days.select([
        "event_date",
        "view_events",
        "purchase_events",
        "purchase_to_view_pct",
        "total_events",
    ])
)

Top 10 days by purchase events:
shape: (10, 5)
┌────────────┬─────────────┬─────────────────┬──────────────────────┬──────────────┐
│ event_date ┆ view_events ┆ purchase_events ┆ purchase_to_view_pct ┆ total_events │
│ ---        ┆ ---         ┆ ---             ┆ ---                  ┆ ---          │
│ str        ┆ u32         ┆ u32             ┆ f64                  ┆ u32          │
╞════════════╪═════════════╪═════════════════╪══════════════════════╪══════════════╡
│ 2019-10-16 ┆ 1445717     ┆ 31394           ┆ 2.172                ┆ 1522390      │
│ 2019-10-13 ┆ 1584677     ┆ 29561           ┆ 1.865                ┆ 1639071      │
│ 2019-10-14 ┆ 1381439     ┆ 28405           ┆ 2.056                ┆ 1454702      │
│ 2019-10-17 ┆ 1338311     ┆ 28318           ┆ 2.116                ┆ 1407007      │
│ 2019-10-04 ┆ 1346320     ┆ 27041           ┆ 2.009                ┆ 1417190      │
│ 2019-10-15 ┆ 1466305     ┆ 26372           ┆ 1.799                ┆ 1542225      │
│ 2019-10-11 ┆ 144

## 18. Statistical Baseline for Daily Purchases

### Objective
Measure the central tendency and variability of daily purchase-event counts across October 2019.

### Method
Calculate the mean, median, standard deviation, quartiles, minimum, and maximum of daily purchase counts using the previously computed daily summary.

### Expected Outcome
A statistical reference for evaluating future synthetic purchase surges.

### Important Notes
The baseline is based on 31 daily observations from one month. It is useful for designing a synthetic scenario, but it is not sufficient to establish recurring seasonal patterns or prove that an observed day represents an anomaly.

In [21]:
purchase_baseline = daily_result.select(
    pl.col("purchase_events").mean().alias("mean_purchases"),
    pl.col("purchase_events").median().alias("median_purchases"),
    pl.col("purchase_events").std().alias("std_purchases"),
    pl.col("purchase_events").quantile(0.25).alias("q1_purchases"),
    pl.col("purchase_events").quantile(0.75).alias("q3_purchases"),
    pl.col("purchase_events").min().alias("min_purchases"),
    pl.col("purchase_events").max().alias("max_purchases"),
)

print(purchase_baseline)

shape: (1, 7)
┌──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ mean_purchas ┆ median_purc ┆ std_purchas ┆ q1_purchase ┆ q3_purchase ┆ min_purchas ┆ max_purchas │
│ es           ┆ hases       ┆ es          ┆ s           ┆ s           ┆ es          ┆ es          │
│ ---          ┆ ---         ┆ ---         ┆ ---         ┆ ---         ┆ ---         ┆ ---         │
│ f64          ┆ f64         ┆ f64         ┆ f64         ┆ f64         ┆ u32         ┆ u32         │
╞══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ 23962.870968 ┆ 23929.0     ┆ 3083.275701 ┆ 21993.0     ┆ 25850.0     ┆ 19255       ┆ 31394       │
└──────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┘


## 19. Cart Abandonment Baseline Analysis

### Objective
Understand cart-to-purchase behavior at the product and session level before defining a synthetic cart-abandonment scenario.

### Method
- Identify cart events for a specific product within a user session.
- Inspect whether a subsequent purchase of the same product occurs within that session.
- Preserve event order when evaluating the behavior.
- Distinguish observed cart abandonment from deliberately injected anomalies.

### Expected Outcome
A baseline definition for measuring cart events that are not followed by a matching purchase within the same session.

### Important Notes
Cart abandonment is common ecommerce behavior and is not inherently a data-quality error. The source dataset contains no observed `remove_from_cart` event type. No original records or schema definitions are modified.

## 19.1 Inspect Cart Events

### Objective
Inspect a small sample of cart events and their corresponding session and product identifiers.

### Method
Select a limited number of cart events from the source dataset, keeping the original columns unchanged.

### Expected Outcome
A sample that can be used to inspect whether each cart event is followed by a purchase of the same product within the same session.

In [22]:
cart_sample = (
    df
    .filter(pl.col("event_type") == "cart")
    .select([
        "event_time",
        "event_type",
        "product_id",
        "user_id",
        "user_session"
    ])
    .head(20)
    .collect()
)

print(cart_sample)

shape: (20, 5)
┌─────────────────────────┬────────────┬────────────┬───────────┬─────────────────────────────────┐
│ event_time              ┆ event_type ┆ product_id ┆ user_id   ┆ user_session                    │
│ ---                     ┆ ---        ┆ ---        ┆ ---       ┆ ---                             │
│ str                     ┆ str        ┆ i64        ┆ i64       ┆ str                             │
╞═════════════════════════╪════════════╪════════════╪═══════════╪═════════════════════════════════╡
│ 2019-10-01 00:05:14 UTC ┆ cart       ┆ 5100816    ┆ 550121407 ┆ 6f623695-9581-4633-813f-825b87… │
│ 2019-10-01 00:09:33 UTC ┆ cart       ┆ 1002524    ┆ 524325294 ┆ 0b74a829-f9d7-4654-b5b0-35bc98… │
│ 2019-10-01 00:11:00 UTC ┆ cart       ┆ 4804056    ┆ 533624186 ┆ e5ac3caa-e6d5-4d6b-ae06-2c18cd… │
│ 2019-10-01 02:17:59 UTC ┆ cart       ┆ 1004833    ┆ 536415846 ┆ 685b5b42-f597-4a69-ab4c-ef96a3… │
│ 2019-10-01 02:19:36 UTC ┆ cart       ┆ 1005003    ┆ 513632293 ┆ f2cc68f7-39d1-4a50-

## 19.2 Inspect Purchases for Sampled Cart Events

### Objective
Check whether products added to a cart are purchased later within the same user session.

### Method
Use the sampled cart records to identify matching purchase events with the same `user_session` and `product_id`, occurring at or after the cart timestamp.

### Expected Outcome
A small table showing cart timestamps and any matching purchase timestamps, without modifying the source dataset.

In [23]:
cart_keys = cart_sample.select([
    "user_session",
    "product_id"
]).unique()

purchase_sample = (
    df
    .filter(pl.col("event_type") == "purchase")
    .join(
        cart_keys.lazy(),
        on=["user_session", "product_id"],
        how="inner"
    )
    .select([
        "event_time",
        "event_type",
        "product_id",
        "user_id",
        "user_session"
    ])
    .head(50)
    .collect()
)

print(purchase_sample)

shape: (8, 5)
┌─────────────────────────┬────────────┬────────────┬───────────┬─────────────────────────────────┐
│ event_time              ┆ event_type ┆ product_id ┆ user_id   ┆ user_session                    │
│ ---                     ┆ ---        ┆ ---        ┆ ---       ┆ ---                             │
│ str                     ┆ str        ┆ i64        ┆ i64       ┆ str                             │
╞═════════════════════════╪════════════╪════════════╪═══════════╪═════════════════════════════════╡
│ 2019-10-01 02:22:29 UTC ┆ purchase   ┆ 1500021    ┆ 513489079 ┆ 63de1120-9d48-45d0-b024-500d1b… │
│ 2019-10-01 02:22:31 UTC ┆ purchase   ┆ 5100719    ┆ 549614436 ┆ 129d379f-0137-4bd5-bf30-90c4c7… │
│ 2019-10-01 00:10:08 UTC ┆ purchase   ┆ 1002524    ┆ 524325294 ┆ 0b74a829-f9d7-4654-b5b0-35bc98… │
│ 2019-10-01 02:32:46 UTC ┆ purchase   ┆ 1005003    ┆ 513632293 ┆ f2cc68f7-39d1-4a50-9dcf-f2a092… │
│ 2019-10-01 02:21:02 UTC ┆ purchase   ┆ 1004659    ┆ 555168263 ┆ d068d08f-aaf3-4949-b

In [25]:
cart_purchase_check = (
    cart_times
    .join(
        purchase_times,
        on=["user_session", "product_id"],
        how="left"
    )
    .filter(
        pl.col("purchase_time").is_not_null()
        & (pl.col("purchase_time") >= pl.col("cart_time"))
    )
    .with_columns(
        (
            pl.col("purchase_time") - pl.col("cart_time")
        ).dt.total_seconds().alias("seconds_to_purchase")
    )
    .group_by(["user_session", "product_id", "cart_time"])
    .agg(
        pl.col("purchase_time").min().alias("first_purchase_after_cart"),
        pl.col("seconds_to_purchase").min().alias("seconds_to_purchase")
    )
    .sort("cart_time")
)

print(cart_purchase_check)

shape: (13, 5)
┌─────────────────────┬────────────┬─────────────────────┬────────────────────┬────────────────────┐
│ user_session        ┆ product_id ┆ cart_time           ┆ first_purchase_aft ┆ seconds_to_purchas │
│ ---                 ┆ ---        ┆ ---                 ┆ er_cart            ┆ e                  │
│ str                 ┆ i64        ┆ datetime[μs, UTC]   ┆ ---                ┆ ---                │
│                     ┆            ┆                     ┆ datetime[μs, UTC]  ┆ i64                │
╞═════════════════════╪════════════╪═════════════════════╪════════════════════╪════════════════════╡
│ 0b74a829-f9d7-4654- ┆ 1002524    ┆ 2019-10-01 00:09:33 ┆ 2019-10-01         ┆ 35                 │
│ b5b0-35bc98…        ┆            ┆ UTC                 ┆ 00:10:08 UTC       ┆                    │
│ f2cc68f7-39d1-4a50- ┆ 1005003    ┆ 2019-10-01 02:19:36 ┆ 2019-10-01         ┆ 790                │
│ 9dcf-f2a092…        ┆            ┆ UTC                 ┆ 02:32:46 UTC     

## 20. Bot-like Crawling Loops — Baseline Analysis

### Objective
Understand the distribution of event counts per user session and identify unusually high-activity sessions for further inspection.

### Method
Use session-level event counts to calculate selected percentiles and inspect the upper tail of the distribution.

### Expected Outcome
A statistical baseline that helps define synthetic bot-like activity without labeling legitimate high-activity users as bots.

### Important Notes
A high event count alone does not prove bot activity. Any injected scenario will be explicitly synthetic and will not modify the original source file or schema.

In [26]:
session_baseline = (
    df
    .filter(pl.col("user_session").is_not_null())
    .group_by("user_session")
    .agg(
        pl.len().alias("event_count")
    )
    .select(
        pl.col("event_count").count().alias("session_count"),
        pl.col("event_count").median().alias("median_events"),
        pl.col("event_count").quantile(0.90).alias("p90_events"),
        pl.col("event_count").quantile(0.95).alias("p95_events"),
        pl.col("event_count").quantile(0.99).alias("p99_events"),
        pl.col("event_count").max().alias("max_events")
    )
    .collect()
)

print(session_baseline)

shape: (1, 6)
┌───────────────┬───────────────┬────────────┬────────────┬────────────┬────────────┐
│ session_count ┆ median_events ┆ p90_events ┆ p95_events ┆ p99_events ┆ max_events │
│ ---           ┆ ---           ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│ u32           ┆ f64           ┆ f64        ┆ f64        ┆ f64        ┆ u32        │
╞═══════════════╪═══════════════╪════════════╪════════════╪════════════╪════════════╡
│ 9244421       ┆ 2.0           ┆ 10.0       ┆ 16.0       ┆ 32.0       ┆ 1159       │
└───────────────┴───────────────┴────────────┴────────────┴────────────┴────────────┘


## 21. Duplicate Events — Baseline Analysis

### Objective
Document exact duplicate records already present in the original dataset before injecting synthetic duplicate events.

### Method
Compare records across all nine original columns and distinguish duplicate groups from extra duplicate rows.

### Expected Outcome
A documented source baseline that prevents pre-existing duplicates from being confused with deliberately injected duplicates.

### Important Notes
No duplicate records are removed. The original file and schema remain unchanged. Synthetic duplicate events will be tracked separately through the injection process, without adding anomaly-label columns to the source schema.

## 22. Synthetic Promotional Purchase Surge — Baseline Analysis

### Objective
Establish a reference for daily purchase-event volume before simulating a promotional surge.

### Method
Use the previously calculated daily purchase statistics to define a synthetic promotional window and compare its purchase volume against the baseline.

### Expected Outcome
A reproducible scenario in which purchase activity rises during a configured period, allowing downstream pipeline and dashboard validation.

### Important Notes
The source dataset covers only October 2019. It does not establish a real promotional calendar or prove that any observed spike was caused by a campaign. The promotion window and surge magnitude will be explicitly synthetic. The original file and schema remain unchanged.